In [ ]:
import matplotlib.pyplot as plt

# 设置支持中文的字体（例如 SimHei），同时确保负号能正常显示
plt.rcParams['font.sans-serif'] = ['Times New Roman', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 绘制基础图

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "Noto Sans CJK SC"]
plt.rcParams["axes.unicode_minus"] = False

# =========================
# 0) 文件路径（改成你自己的）
# =========================
CSV_PATH = "../data/7.匹配时间/merged_output.csv"

PROV_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_省.geojson"
CITY_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_市.geojson"
COUNTY_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_县.geojson"


# =========================
# 1) 读点数据（WGS84 / EPSG:4326）
# =========================
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["lng", "lat"]).copy()
df["lng"] = df["lng"].astype(float)
df["lat"] = df["lat"].astype(float)

gdf_pt = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lng"], df["lat"]),
    crs="EPSG:4326"
)

#（可选）粗略过滤明显异常坐标（避免把点落到国外/海上）
gdf_pt = gdf_pt[(gdf_pt["lng"].between(70, 140)) & (gdf_pt["lat"].between(3, 55))].copy()


# =========================
# 2) 读天地图边界（EPSG:4490），统一转成 EPSG:4326
# =========================
def read_tianditu_geojson(path):
    gdf = gpd.read_file(path)

    # 天地图 GeoJSON 里写的是 EPSG:4490；有时 geopandas 读出来 crs 可能是 None
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4490", allow_override=True)

    # 统一到 WGS84（EPSG:4326）以便和点数据一致
    gdf = gdf.to_crs("EPSG:4326")

    # 修复可能存在的无效几何（避免 sjoin 报错）
    try:
        gdf["geometry"] = gdf["geometry"].make_valid()
    except Exception:
        gdf["geometry"] = gdf["geometry"].buffer(0)

    return gdf


adm1 = read_tianditu_geojson(PROV_GEOJSON)    # 省
adm2 = read_tianditu_geojson(CITY_GEOJSON)    # 市
adm3 = read_tianditu_geojson(COUNTY_GEOJSON)  # 县/区


# =========================
# 3) 点落面并统计
# =========================
def spatial_count(points_gdf, poly_gdf, key_col="gb", name_col="name", predicate="within"):
    """
    predicate:
      - 'within'：点必须严格在面内（最常用）
      - 'intersects'：点在边界上也算（如果你担心边界点丢失，可用这个）
    """
    cols = [c for c in [key_col, name_col] if c in poly_gdf.columns]
    cols = cols + ["geometry"]

    # 空间连接：每个点落到哪个行政区
    joined = gpd.sjoin(
        points_gdf[["geometry"]],
        poly_gdf[cols],
        how="left",
        predicate=predicate
    )

    # 用 gb（行政区划代码）统计更稳，避免重名
    if key_col in poly_gdf.columns:
        cnt = joined.groupby(key_col).size().rename("n").reset_index()
        out = poly_gdf.merge(cnt, on=key_col, how="left")
    else:
        # 如果你的 geojson 没有 gb，就退化用 name 统计
        cnt = joined.groupby(name_col).size().rename("n").reset_index()
        out = poly_gdf.merge(cnt, on=name_col, how="left")

    out["n"] = out["n"].fillna(0).astype(int)
    return out


adm1_cnt = spatial_count(gdf_pt, adm1, key_col="gb", name_col="name", predicate="within")
adm2_cnt = spatial_count(gdf_pt, adm2, key_col="gb", name_col="name", predicate="within")
adm3_cnt = spatial_count(gdf_pt, adm3, key_col="gb", name_col="name", predicate="within")


# =========================
# 4) 绘图：三尺度分级设色（带 colorbar）
# =========================
def plot_choropleth(gdf, title, figsize=(7.5, 6), linewidth=0.2):
    fig, ax = plt.subplots(figsize=figsize)

    # ✅ 仅用于显示：转到 Web Mercator (EPSG:3857)
    # gdf = gdf.to_crs("EPSG:3857")
    
    # 让 colorbar 更像论文图：连续色带 + 标题
    gdf.plot(
        column="n",
        ax=ax,
        cmap="Spectral_r",      # 蓝->红（你也可以换成别的）
        linewidth=linewidth,
        edgecolor="0.7",
        legend=True,
        missing_kwds={"color": "white"},
        legend_kwds={"label": "Number of Case", "shrink": 0.6}
    )

    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


plot_choropleth(adm1_cnt, "(a) Province-level", figsize=(7.5, 6), linewidth=0.5)
plot_choropleth(adm2_cnt, "(b) City-level",     figsize=(7.5, 6), linewidth=0.15)
plot_choropleth(adm3_cnt, "(c) County-level",   figsize=(7.5, 6), linewidth=0.05)


In [ ]:
# =========================
# 5) 统计有数据的省市县数量
# =========================
def print_data_coverage(gdf_cnt, level_name):
    """
    统计并打印某一级行政区的覆盖情况
    """
    total_count = len(gdf_cnt)
    # 筛选出有点落入（n > 0）的行政区
    with_data_count = len(gdf_cnt[gdf_cnt["n"] > 0])
    no_data_count = total_count - with_data_count
    
    # 计算覆盖率
    coverage_rate = (with_data_count / total_count) * 100 if total_count > 0 else 0

    print(f"[{level_name}] 统计:")
    print(f"  - 行政区总数: {total_count}")
    print(f"  - 有数据的数量: {with_data_count}")
    print(f"  - 无数据的数量: {no_data_count}")
    print(f"  - 数据覆盖率: {coverage_rate:.2f}%\n")

print("\n================ 数据落点覆盖率报告 ================")
print_data_coverage(adm1_cnt, "省级 (Province-level)")
print_data_coverage(adm2_cnt, "市级 (City-level)")
print_data_coverage(adm3_cnt, "县/区级 (County-level)")
print("==================================================\n")

# 修改绘图增加经纬度线

pip install cartopy

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# 方案A需要 cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "Noto Sans CJK SC"]
plt.rcParams["axes.unicode_minus"] = False

# =========================
# 0) 文件路径（改成你自己的）
# =========================
CSV_PATH = "../data/7.匹配时间/merged_output.csv"

PROV_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_省.geojson"
CITY_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_市.geojson"
COUNTY_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_县.geojson"


# =========================
# 1) 读点数据（WGS84 / EPSG:4326）
# =========================
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["lng", "lat"]).copy()
df["lng"] = df["lng"].astype(float)
df["lat"] = df["lat"].astype(float)

gdf_pt = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lng"], df["lat"]),
    crs="EPSG:4326")

# （可选）粗略过滤异常坐标
gdf_pt = gdf_pt[(gdf_pt["lng"].between(70, 140)) & (gdf_pt["lat"].between(3, 55))].copy()


# =========================
# 2) 读边界（天地图常见 EPSG:4490）-> 统一转 EPSG:4326
# =========================
def read_tianditu_geojson(path):
    gdf = gpd.read_file(path)

    # 天地图 GeoJSON 常写 EPSG:4490；有时读出来 crs=None
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4490", allow_override=True)

    # 统一到 WGS84 便于和点数据空间连接
    gdf = gdf.to_crs("EPSG:4326")

    # 修复几何
    try:
        # shapely>=2
        gdf["geometry"] = gdf["geometry"].make_valid()
    except Exception:
        gdf["geometry"] = gdf["geometry"].buffer(0)

    return gdf


adm1 = read_tianditu_geojson(PROV_GEOJSON)    # 省
adm2 = read_tianditu_geojson(CITY_GEOJSON)    # 市
adm3 = read_tianditu_geojson(COUNTY_GEOJSON)  # 县/区


# =========================
# 3) 点落面并统计
# =========================
def spatial_count(points_gdf, poly_gdf, key_col="gb", name_col="name", predicate="within"):
    cols = [c for c in [key_col, name_col] if c in poly_gdf.columns]
    cols = cols + ["geometry"]

    joined = gpd.sjoin(
        points_gdf[["geometry"]],
        poly_gdf[cols],
        how="left",
        predicate=predicate)

    if key_col in poly_gdf.columns:
        cnt = joined.groupby(key_col).size().rename("n").reset_index()
        out = poly_gdf.merge(cnt, on=key_col, how="left")
    else:
        cnt = joined.groupby(name_col).size().rename("n").reset_index()
        out = poly_gdf.merge(cnt, on=name_col, how="left")

    out["n"] = out["n"].fillna(0).astype(int)
    return out


adm1_cnt = spatial_count(gdf_pt, adm1, key_col="gb", name_col="name", predicate="within")
adm2_cnt = spatial_count(gdf_pt, adm2, key_col="gb", name_col="name", predicate="within")
adm3_cnt = spatial_count(gdf_pt, adm3, key_col="gb", name_col="name", predicate="within")


# =========================
# 4) 制图：Cartopy 圆锥投影 + 弯曲经纬网
# =========================
# 推荐：Albers 等积圆锥（中国常用）
# 这套参数会产生你参考图那种“经纬网弯曲”的效果
CHINA_ALBERS = ccrs.AlbersEqualArea(
    central_longitude=105,
    standard_parallels=(25, 47)
)

# 为了让 GeoPandas 的 to_crs 和 Cartopy 投影一致：
# 用同一套 PROJ4 参数（GeoPandas/pyproj 能稳定识别）
ALBERS_PROJ4 = (
    "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105 "
    "+datum=WGS84 +units=m +no_defs"
)

def plot_choropleth_cartopy(gdf, panel_title, figsize=(7.5, 6), linewidth=0.2,
                            extent=(73, 135, 18, 54),
                            grid_step=(10, 10),
                            cmap="Spectral_r"):
    """
    extent: (min_lon, max_lon, min_lat, max_lat) 以经纬度给范围
    grid_step: (dlon, dlat) 经线/纬线间隔（度）
    """
    fig = plt.figure(figsize=figsize, dpi=300)
    ax = plt.axes(projection=CHINA_ALBERS)

    # 范围（用经纬度定义，再告诉 cartopy 这是 PlateCarree）
    ax.set_extent([extent[0], extent[1], extent[2], extent[3]], crs=ccrs.PlateCarree())

    # 可选：加底图要素（不想要可以注释）
    ax.add_feature(cfeature.LAND.with_scale("50m"), linewidth=0, zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), linewidth=0, zorder=0)

    # ✅ 弯曲经纬网 + 标签（最接近参考图）
    dlon, dlat = grid_step
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        xlocs=np.arange(int(extent[0]//dlon*dlon), extent[1] + 0.1, dlon),
        ylocs=np.arange(int(extent[2]//dlat*dlat), extent[3] + 0.1, dlat),
        linewidth=0.4,
        color="0.85",
        linestyle="-",
        zorder=1
    )
    gl.top_labels = False
    gl.right_labels = False
    
    # ✅ 关键：关闭自动旋转（让经纬度标签横着）
    try:
        gl.rotate_labels = False   # cartopy 新版本有效
    except Exception:
        pass
    
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    # ✅ 再保险：强制 rotation=0
    gl.xlabel_style = {"size": 9, "color": "0.25", "rotation": 0, "ha": "center", "va": "center"}
    gl.ylabel_style = {"size": 9, "color": "0.25", "rotation": 0, "ha": "right",  "va": "center"}
    
    # （可选）如果觉得标签贴得太近/太远，可以调这个
    # gl.xpadding = 5
    # gl.ypadding = 5

    # ✅ 把行政区投影到 Albers（单位：米），再用 GeoPandas 直接画到 cartopy 的 ax 上
    gdfp = gdf.to_crs(ALBERS_PROJ4)
    

    gdfp.plot(
        column="n",
        ax=ax,
        cmap=cmap,
        linewidth=linewidth,
        edgecolor="0.7",
        legend=True,
        missing_kwds={"color": "white"},
        legend_kwds={
            "label": "Number of Case",
            "shrink": 0.6,
            "orientation": "vertical"},
        zorder=2
    )

    # panel 标注：像参考图那样放左上角
    ax.text(
        0.02, 0.98, panel_title,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=12)

    # 去掉外框
    # ax.set_axis_off()
    plt.tight_layout()
    plt.show()


# =========================
# 5) 画三张图
# =========================
plot_choropleth_cartopy(adm1_cnt, "(a) Province Level", figsize=(7.5, 6), linewidth=0.5, grid_step=(10, 10))
plot_choropleth_cartopy(adm2_cnt, "(b) City Level",     figsize=(7.5, 6), linewidth=0.15, grid_step=(10, 10))
plot_choropleth_cartopy(adm3_cnt, "(c) County Level",   figsize=(7.5, 6), linewidth=0.05, grid_step=(10, 10))


# 修改绘图增加南海区域

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# 方案A需要 cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "Noto Sans CJK SC"]
plt.rcParams["axes.unicode_minus"] = False

# =========================
# 0) 文件路径（改成你自己的）
# =========================
CSV_PATH = "../data/7.匹配时间/merged_output.csv"

PROV_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_省.geojson"
CITY_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_市.geojson"
COUNTY_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_县.geojson"

# =========================
# 1) 读点数据（WGS84 / EPSG:4326）
# =========================
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["lng", "lat"]).copy()
df["lng"] = df["lng"].astype(float)
df["lat"] = df["lat"].astype(float)

gdf_pt = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lng"], df["lat"]),
    crs="EPSG:4326"
)

# （可选）粗略过滤异常坐标：南海要显示的话，纬度下限建议到 0 或 3
gdf_pt = gdf_pt[(gdf_pt["lng"].between(70, 140)) & (gdf_pt["lat"].between(0, 56))].copy()

# =========================
# 2) 读边界（天地图常见 EPSG:4490）-> 统一转 EPSG:4326
# =========================
def read_tianditu_geojson(path: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)

    # 天地图 GeoJSON 常写 EPSG:4490；有时读出来 crs=None
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4490", allow_override=True)

    # 统一到 WGS84 便于和点数据空间连接
    gdf = gdf.to_crs("EPSG:4326")

    # 修复几何
    try:
        # shapely>=2
        gdf["geometry"] = gdf["geometry"].make_valid()
    except Exception:
        gdf["geometry"] = gdf["geometry"].buffer(0)

    return gdf


adm1 = read_tianditu_geojson(PROV_GEOJSON)    # 省
adm2 = read_tianditu_geojson(CITY_GEOJSON)    # 市
adm3 = read_tianditu_geojson(COUNTY_GEOJSON)  # 县/区

# ✅ 融合出“中国整体外边界”，用于画黑色国界线（不会显示内部省界）
china_outline = adm1[["geometry"]].dissolve()
china_outline = china_outline.set_crs(adm1.crs)

# =========================
# 3) 点落面并统计
# =========================
def spatial_count(points_gdf, poly_gdf, key_col="gb", name_col="name", predicate="within"):
    cols = [c for c in [key_col, name_col] if c in poly_gdf.columns]
    cols = cols + ["geometry"]

    joined = gpd.sjoin(
        points_gdf[["geometry"]],
        poly_gdf[cols],
        how="left",
        predicate=predicate
    )

    if key_col in poly_gdf.columns:
        cnt = joined.groupby(key_col).size().rename("n").reset_index()
        out = poly_gdf.merge(cnt, on=key_col, how="left")
    else:
        cnt = joined.groupby(name_col).size().rename("n").reset_index()
        out = poly_gdf.merge(cnt, on=name_col, how="left")

    out["n"] = out["n"].fillna(0).astype(int)
    return out


adm1_cnt = spatial_count(gdf_pt, adm1, key_col="gb", name_col="name", predicate="within")
adm2_cnt = spatial_count(gdf_pt, adm2, key_col="gb", name_col="name", predicate="within")
adm3_cnt = spatial_count(gdf_pt, adm3, key_col="gb", name_col="name", predicate="within")

# =========================
# 4) 制图：Cartopy 圆锥投影 + 弯曲经纬网
# =========================
CHINA_ALBERS = ccrs.AlbersEqualArea(
    central_longitude=105,
    standard_parallels=(25, 47)
)

ALBERS_PROJ4 = (
    "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105 "
    "+datum=WGS84 +units=m +no_defs"
)

def plot_choropleth_cartopy(
    gdf,
    panel_title,
    figsize=(7.5, 6),
    extent=(73, 136, 0, 55),      # ✅ 主图直接包含南海区域（南到 0°N）
    grid_step=(10, 10),
    cmap="Spectral_r",
    inner_linewidth=0.15,         # ✅ 省/市/县内部边界线宽（你要缩小的就是这个）
    outer_linewidth=1.0,          # ✅ 中国国界线黑色线宽
    inner_edgecolor="0.65"        # 内部边界颜色（灰）
):
    """
    extent: (min_lon, max_lon, min_lat, max_lat) 以经纬度给范围
    """
    fig = plt.figure(figsize=figsize, dpi=300)
    ax = plt.axes(projection=CHINA_ALBERS)

    # 范围（用经纬度定义，再告诉 cartopy 这是 PlateCarree）
    ax.set_extent([extent[0], extent[1], extent[2], extent[3]], crs=ccrs.PlateCarree())

    # 底图要素（可选）
    ax.add_feature(cfeature.LAND.with_scale("50m"), linewidth=0, zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), linewidth=0, zorder=0)

    # 弯曲经纬网 + 标签
    dlon, dlat = grid_step
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        xlocs=np.arange(int(extent[0]//dlon*dlon), extent[1] + 0.1, dlon),
        ylocs=np.arange(int(extent[2]//dlat*dlat), extent[3] + 0.1, dlat),
        linewidth=0.4,
        color="0.85",
        linestyle="-",
        zorder=1
    )
    gl.top_labels = False
    gl.right_labels = False

    # ✅ 关键：关闭自动旋转（让经纬度标签横着）
    try:
        gl.rotate_labels = False   # cartopy 新版本有效
    except Exception:
        pass
    
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    # ✅ 再保险：强制 rotation=0
    gl.xlabel_style = {"size": 9, "color": "0.25", "rotation": 0, "ha": "center", "va": "center"}
    gl.ylabel_style = {"size": 9, "color": "0.25", "rotation": 0, "ha": "right",  "va": "center"}
    
    # （可选）如果觉得标签贴得太近/太远，可以调这个
    # gl.xpadding = 5
    # gl.ypadding = 5


    # ✅ 1) 先画分级填色 + 内部边界（更细）
    gdfp = gdf.to_crs(ALBERS_PROJ4)
    gdfp.plot(
        column="n",
        ax=ax,
        cmap=cmap,
        linewidth=inner_linewidth,
        edgecolor=inner_edgecolor,
        legend=True,
        missing_kwds={"color": "white"},
        legend_kwds={"label": "Number of Case", "shrink": 0.6, "orientation": "vertical"},
        zorder=2
    )

    # ✅ 2) 再叠加画中国国界线（黑色）
    outlinep = china_outline.to_crs(ALBERS_PROJ4)
    outlinep.boundary.plot(
        ax=ax,
        color="black",
        linewidth=outer_linewidth,
        zorder=4
    )

    # panel 标注：左上角
    ax.text(
        0.02, 0.98, panel_title,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=12
    )

    plt.tight_layout()
    plt.show()


# =========================
# 5) 画三张图（内部边界线宽更细；国界线黑色）
# =========================
plot_choropleth_cartopy(
    adm1_cnt, "(a) Province Level",
    figsize=(7.5, 6),
    extent=(73, 136, 0, 55),
    grid_step=(10, 10),
    inner_linewidth=0.18,
    outer_linewidth=1.1
)

plot_choropleth_cartopy(
    adm2_cnt, "(b) City Level",
    figsize=(7.5, 6),
    extent=(73, 136, 0, 55),
    grid_step=(10, 10),
    inner_linewidth=0.06,  # ✅ 市界更细
    outer_linewidth=1.1
)

plot_choropleth_cartopy(
    adm3_cnt, "(c) County Level",
    figsize=(7.5, 6),
    extent=(73, 136, 0, 55),
    grid_step=(10, 10),
    inner_linewidth=0.02,  # ✅ 县界最细
    outer_linewidth=1.1
)


# 修改底图风格

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# 方案A需要 cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "Noto Sans CJK SC"]
plt.rcParams["axes.unicode_minus"] = False

# =========================
# 0) 文件路径（改成你自己的）
# =========================
CSV_PATH = "../data/7.匹配时间/merged_output.csv"

PROV_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_省.geojson"
CITY_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_市.geojson"
COUNTY_GEOJSON = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_县.geojson"

# =========================
# 1) 读点数据（WGS84 / EPSG:4326）
# =========================
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["lng", "lat"]).copy()
df["lng"] = df["lng"].astype(float)
df["lat"] = df["lat"].astype(float)

gdf_pt = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lng"], df["lat"]),
    crs="EPSG:4326"
)

# （可选）粗略过滤异常坐标：南海要显示的话，纬度下限建议到 0 或 3
gdf_pt = gdf_pt[(gdf_pt["lng"].between(70, 140)) & (gdf_pt["lat"].between(0, 56))].copy()

# =========================
# 2) 读边界（天地图常见 EPSG:4490）-> 统一转 EPSG:4326
# =========================
def read_tianditu_geojson(path: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)

    # 天地图 GeoJSON 常写 EPSG:4490；有时读出来 crs=None
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4490", allow_override=True)

    # 统一到 WGS84 便于和点数据空间连接
    gdf = gdf.to_crs("EPSG:4326")

    # 修复几何
    try:
        # shapely>=2
        gdf["geometry"] = gdf["geometry"].make_valid()
    except Exception:
        gdf["geometry"] = gdf["geometry"].buffer(0)

    return gdf


adm1 = read_tianditu_geojson(PROV_GEOJSON)    # 省
adm2 = read_tianditu_geojson(CITY_GEOJSON)    # 市
adm3 = read_tianditu_geojson(COUNTY_GEOJSON)  # 县/区

# ✅ 融合出“中国整体外边界”，用于画黑色国界线（不会显示内部省界）
china_outline = adm1[["geometry"]].dissolve()
china_outline = china_outline.set_crs(adm1.crs)

# =========================
# 3) 点落面并统计
# =========================
def spatial_count(points_gdf, poly_gdf, key_col="gb", name_col="name", predicate="within"):
    cols = [c for c in [key_col, name_col] if c in poly_gdf.columns]
    cols = cols + ["geometry"]

    joined = gpd.sjoin(
        points_gdf[["geometry"]],
        poly_gdf[cols],
        how="left",
        predicate=predicate
    )

    if key_col in poly_gdf.columns:
        cnt = joined.groupby(key_col).size().rename("n").reset_index()
        out = poly_gdf.merge(cnt, on=key_col, how="left")
    else:
        cnt = joined.groupby(name_col).size().rename("n").reset_index()
        out = poly_gdf.merge(cnt, on=name_col, how="left")

    out["n"] = out["n"].fillna(0).astype(int)
    return out


adm1_cnt = spatial_count(gdf_pt, adm1, key_col="gb", name_col="name", predicate="within")
adm2_cnt = spatial_count(gdf_pt, adm2, key_col="gb", name_col="name", predicate="within")
adm3_cnt = spatial_count(gdf_pt, adm3, key_col="gb", name_col="name", predicate="within")

# =========================
# 4) 制图：Cartopy 圆锥投影 + 弯曲经纬网
# =========================
CHINA_ALBERS = ccrs.AlbersEqualArea(
    central_longitude=105,
    standard_parallels=(25, 47)
)

ALBERS_PROJ4 = (
    "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105 "
    "+datum=WGS84 +units=m +no_defs"
)

def plot_choropleth_cartopy(
    gdf,
    panel_title,
    figsize=(7.5, 6),
    extent=(73, 136, 0, 55),      # ✅ 主图直接包含南海区域（南到 0°N）
    grid_step=(10, 10),
    cmap="Spectral_r",
    inner_linewidth=0.15,         # ✅ 省/市/县内部边界线宽
    outer_linewidth=1.0,          # ✅ 中国国界线宽（黑色）
    inner_edgecolor="0.65",       # 内部边界颜色（灰）
    basemap_style="paper",        # ✅ 底图风格：paper / blue / gray
    show_coast_river=True         # ✅ 是否叠加海岸线/河流/湖泊
):
    fig = plt.figure(figsize=figsize, dpi=300)
    ax = plt.axes(projection=CHINA_ALBERS)

    # 范围（经纬度范围 + PlateCarree）
    ax.set_extent([extent[0], extent[1], extent[2], extent[3]], crs=ccrs.PlateCarree())

    # =========================
    # 底图样式（你要的三种：纯白 / 蓝海米地 / 灰度）
    # =========================
    if basemap_style == "paper":
        # 纯白简洁论文风
        ax.set_facecolor("white")
    elif basemap_style == "blue":
        # 蓝色海洋 + 米色陆地
        land = cfeature.NaturalEarthFeature(
            "physical", "land", "50m",
            edgecolor="none", facecolor="#f3f1e7"
        )
        ocean = cfeature.NaturalEarthFeature(
            "physical", "ocean", "50m",
            edgecolor="none", facecolor="#dbe9f6"
        )
        ax.add_feature(ocean, zorder=0)
        ax.add_feature(land, zorder=0.1)
    elif basemap_style == "gray":
        # 灰度底图
        land = cfeature.NaturalEarthFeature(
            "physical", "land", "50m",
            edgecolor="none", facecolor="0.93"
        )
        ocean = cfeature.NaturalEarthFeature(
            "physical", "ocean", "50m",
            edgecolor="none", facecolor="0.98"
        )
        ax.add_feature(ocean, zorder=0)
        ax.add_feature(land, zorder=0.1)
    else:
        # 默认：纯白
        ax.set_facecolor("white")

    # ✅ 海岸线 + 河流 + 湖泊（可选叠加）
    if show_coast_river:
        ax.coastlines(resolution="50m", linewidth=0.6, zorder=1)
        ax.add_feature(
            cfeature.RIVERS.with_scale("50m"),
            edgecolor="0.6", linewidth=0.3, zorder=1
        )
        ax.add_feature(
            cfeature.LAKES.with_scale("50m"),
            facecolor="none", edgecolor="0.6", linewidth=0.3, zorder=1
        )

    # =========================
    # 弯曲经纬网 + 标签
    # =========================
    dlon, dlat = grid_step
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        xlocs=np.arange(int(extent[0]//dlon*dlon), extent[1] + 0.1, dlon),
        ylocs=np.arange(int(extent[2]//dlat*dlat), extent[3] + 0.1, dlat),
        linewidth=0.4,
        color="0.85",
        linestyle="-",
        zorder=2
    )
    gl.top_labels = False
    gl.right_labels = False
    
    # ✅ 关键：关闭自动旋转（让经纬度标签横着）
    try:
        gl.rotate_labels = False   # cartopy 新版本有效
    except Exception:
        pass
    
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    # ✅ 再保险：强制 rotation=0
    gl.xlabel_style = {"size": 9, "color": "0.25", "rotation": 0, "ha": "center", "va": "center"}
    gl.ylabel_style = {"size": 9, "color": "0.25", "rotation": 0, "ha": "right",  "va": "center"}
    
    # （可选）如果觉得标签贴得太近/太远，可以调这个
    # gl.xpadding = 5
    # gl.ypadding = 5
    

    # =========================
    # 先画分级填色 + 内部边界（更细）
    # =========================
    gdfp = gdf.to_crs(ALBERS_PROJ4)
    gdfp.plot(
        column="n",
        ax=ax,
        cmap=cmap,
        linewidth=inner_linewidth,
        edgecolor=inner_edgecolor,
        legend=True,
        missing_kwds={"color": "white"},
        legend_kwds={"label": "Number of Case", "shrink": 0.6, "orientation": "vertical"},
        zorder=3
    )

    # =========================
    # 再叠加画中国国界线（黑色）
    # =========================
    outlinep = china_outline.to_crs(ALBERS_PROJ4)
    outlinep.boundary.plot(
        ax=ax,
        color="black",
        linewidth=outer_linewidth,
        zorder=5
    )

    # panel 标注：左上角
    ax.text(
        0.02, 0.98, panel_title,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=12
    )

    plt.tight_layout()
    plt.show()


# =========================
# 5) 画三张图
# 你可以在这里切换底图风格 basemap_style:
#   "paper" = 纯白简洁论文风
#   "blue"  = 蓝色海洋+米色陆地
#   "gray"  = 灰度底图
# =========================
BASEMAP = "blue"   # ← 在这里改成 "blue" 或 "gray"

plot_choropleth_cartopy(
    adm1_cnt, "(a) Province Level",
    figsize=(7.5, 6),
    extent=(73, 136, 0, 55),
    grid_step=(10, 10),
    inner_linewidth=0.18,
    outer_linewidth=1.1,
    basemap_style=BASEMAP,
    show_coast_river=True
)

plot_choropleth_cartopy(
    adm2_cnt, "(b) City Level",
    figsize=(7.5, 6),
    extent=(73, 136, 0, 55),
    grid_step=(10, 10),
    inner_linewidth=0.06,  # ✅ 市界更细
    outer_linewidth=1.1,
    basemap_style=BASEMAP,
    show_coast_river=True
)

plot_choropleth_cartopy(
    adm3_cnt, "(c) County Level",
    figsize=(7.5, 6),
    extent=(73, 136, 0, 55),
    grid_step=(10, 10),
    inner_linewidth=0.02,  # ✅ 县界最细
    outer_linewidth=1.1,
    basemap_style=BASEMAP,
    show_coast_river=True
)


# 修改南海小图位置

In [ ]:
# === 新增以下全局字体设置 ===
plt.rcParams["font.size"] = 12          # 全局默认字体大小（将作用于图例刻度等）
plt.rcParams["axes.titlesize"] = 14     # 子图标题大小
plt.rcParams["axes.labelsize"] = 12     # 坐标轴标签大小

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "Noto Sans CJK SC"]
plt.rcParams["axes.unicode_minus"] = False

# 在上方读点

# =========================
# 4) 制图：修改后的函数
# =========================
CHINA_ALBERS = ccrs.AlbersEqualArea(
    central_longitude=105,
    standard_parallels=(25, 47)
)

ALBERS_PROJ4 = (
    "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105 "
    "+datum=WGS84 +units=m +no_defs"
)

def plot_choropleth_cartopy(
    gdf,
    panel_title,
    figsize=(7.5, 6),
    extent=(73, 136, 18, 54),     # 主图默认纬度改成 18-54，聚焦大陆
    grid_step=(10, 10),
    cmap="Spectral_r",
    inner_linewidth=0.15,
    outer_linewidth=1.1,
    inner_edgecolor="0.65",
    basemap_style="paper",
    show_coast_river=True,
    save_path=''
):
    fig = plt.figure(figsize=figsize, dpi=300)
    ax = plt.axes(projection=CHINA_ALBERS)

    # 1. 设置主图范围
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    # 2. 设置底图风格 (逻辑保持不变)
    if basemap_style == "paper":
        ax.set_facecolor("white")
    elif basemap_style == "blue":
        land = cfeature.NaturalEarthFeature("physical", "land", "50m", edgecolor="none", facecolor="#f3f1e7")
        ocean = cfeature.NaturalEarthFeature("physical", "ocean", "50m", edgecolor="none", facecolor="#dbe9f6")
        ax.add_feature(ocean, zorder=0, alpha=1)  # alpha调整透明度
        ax.add_feature(land, zorder=0.1, alpha=1)
    elif basemap_style == "gray":
        land = cfeature.NaturalEarthFeature("physical", "land", "50m", edgecolor="none", facecolor="0.93")
        ocean = cfeature.NaturalEarthFeature("physical", "ocean", "50m", edgecolor="none", facecolor="0.98")
        ax.add_feature(ocean, zorder=0, alpha=1)
        ax.add_feature(land, zorder=0.1, alpha=1)

    # 3. 海岸线河流 / 此处的linewidth控制饿海岸线粗细
    if show_coast_river:
        ax.coastlines(resolution="50m", linewidth=0.1, zorder=1)
        ax.add_feature(cfeature.RIVERS.with_scale("50m"), edgecolor="0.6", linewidth=0.3, zorder=1)
        ax.add_feature(cfeature.LAKES.with_scale("50m"), facecolor="none", edgecolor="0.6", linewidth=0.3, zorder=1)

    # 4. 经纬网
    dlon, dlat = grid_step
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        xlocs=np.arange(int(extent[0]//dlon*dlon), extent[1] + 0.1, dlon),
        ylocs=np.arange(int(extent[2]//dlat*dlat), extent[3] + 0.1, dlat),
        linewidth=0.4, color="0.85", linestyle="-", zorder=2
    )
    gl.top_labels = False
    gl.right_labels = False
    try:
        gl.rotate_labels = False
    except Exception:
        pass
    gl.xlabel_style = {"size": 12, "color": "0.25", "rotation": 0}
    gl.ylabel_style = {"size": 12, "color": "0.25", "rotation": 0}

    # 5. 主图数据绘制
    gdfp = gdf.to_crs(ALBERS_PROJ4)
    # 计算 Legend 的范围，保证主图和附图颜色一致
    vmin, vmax = gdfp["n"].min(), gdfp["n"].max()

    # 【修改核心】：手动添加一个坐标轴来放图例
    # 参数对应 Figure 坐标系：[左(x), 底(y), 宽(w), 高(h)]
    # x=0.88: 放在主图右侧
    # y=0.25, h=0.5: 高度占一半，居中偏下，比原来更小
    # w=0.015: 控制图例的粗细
    cax = fig.add_axes([0.87, 0.40, 0.015, 0.4])
    
    gdfp.plot(
            column="n", 
            ax=ax, 
            cax=cax,            # <--- 关键：把自定义的坐标轴传进去
            cmap=cmap,
            linewidth=inner_linewidth, 
            edgecolor=inner_edgecolor,
            legend=True,
            vmin=vmin, vmax=vmax,
            missing_kwds={"color": "white"},
            # 注意：使用 cax 后，shrink/fraction 等位置参数失效，只需保留 label 即可
            legend_kwds={"label": "Number of Case"}, 
            zorder=3)

    outlinep = china_outline.to_crs(ALBERS_PROJ4)
    outlinep.boundary.plot(ax=ax, color="black", linewidth=outer_linewidth, zorder=5)

    ax.text(0.02, 0.98, panel_title, transform=ax.transAxes, ha="left", va="top", fontsize=12)

    # =======================================================
    # 【新增】添加南海小图 (Inset Map)
    # =======================================================
    # 1. 创建子图位置：[左, 底, 宽, 高] [x, y, width, height](基于 Figure 坐标 0-1)
    # 这些数值可以根据需要微调
    sub_rect = [0.69, 0.12, 0.4, 0.20] 
    ax_sub = fig.add_axes(sub_rect, projection=CHINA_ALBERS)

    # 2. 设置南海范围 (经度 106-123, 纬度 2-24)
    ax_sub.set_extent([106, 122, 2, 24], crs=ccrs.PlateCarree())

    # 3. 绘制相同的数据 (复用 gdfp 和 outlinep)
    # 注意：这里不加 legend，也不加 gridlines 以保持整洁
    if basemap_style == "paper":
        ax_sub.set_facecolor("white")
    # 如果需要蓝色背景需再次添加 feature，这里简化处理只填充背景色
    
    # 绘制填色
    gdfp.plot(
        column="n", ax=ax_sub, cmap=cmap,
        linewidth=inner_linewidth, edgecolor=inner_edgecolor,
        vmin=vmin, vmax=vmax, # 保持与主图色标一致
        missing_kwds={"color": "white"},
        zorder=3)
    # 绘制国界
    outlinep.boundary.plot(ax=ax_sub, color="black", linewidth=outer_linewidth, zorder=5)

    # 4. 给小图加个边框 (Spine)
    for spine in ax_sub.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(0.8)

    plt.tight_layout() # 注意：tight_layout 有时会警告，因为手动加了 axes，忽略即可

    # =========================
    # 【新增2】 保存图片的逻辑
    # =========================
    if save_path:
        plt.savefig(
            save_path, 
            dpi=300,             # 分辨率：论文通常要求 300 或 600
            bbox_inches='tight', # 自动裁剪掉周围多余的白边
            pad_inches=0.1       # 裁剪后保留一点点边距
        )
        print(f"图片已保存至: {save_path}")

    plt.show() # show 必须在 savefig 之后


# =========================
# 5) 画三张图 (调用参数微调)
# extent = (最小经度, 最大经度, 最小纬度, 最大纬度)
# =========================
# 可以在这里切换底图风格 basemap_style:
#   "paper" = 纯白简洁论文风
#   "blue"  = 蓝色海洋+米色陆地
#   "gray"  = 灰度底图
# =========================
BASEMAP = "white"

# 注意：extent 参数修改为聚焦大陆 (纬度 18 起)
plot_choropleth_cartopy(
    adm1_cnt, "(a) Province Level",
    extent=(73, 132, 15, 54),   # <--- 修改了这里
    inner_linewidth=0.18, 
    outer_linewidth=1.1, basemap_style=BASEMAP,
    save_path="../data/figure/Level Province.png"  # <--- 指定保存路径
)

plot_choropleth_cartopy(
    adm2_cnt, "(b) City Level",
    extent=(73, 132, 15, 54),   # <--- 修改了这里
    inner_linewidth=0.06, 
    outer_linewidth=1.1, basemap_style=BASEMAP,
    save_path="../data/figure/Level City.png"  # <--- 指定保存路径
)

plot_choropleth_cartopy(
    adm3_cnt, "(c) District Level",
    extent=(73, 132, 15, 54),   # <--- 修改了这里
    inner_linewidth=0.02, 
    outer_linewidth=1.1, basemap_style=BASEMAP,
    save_path="../data/figure/Level District.png"  # <--- 指定保存路径
)